# NB4 — LOSO: Janela Ótima, Nível Ótimo e Comparação de Modelos

**Pipeline de Predição de Crises Epilépticas a partir de EEG**

## Fluxo

```
NB3 → data/features/{dataset}__{pat}.npz   (X_pre, X_inter, t_pre, t_inter, ctx_ids)
NB2 → data/preictal_estimate.json          (PRE_SEC individual por paciente)
                          ↓
NB4 → Etapa 1: qual janela W performa melhor?      (livre=W, fixo=nível R5/máx, modelo=RF)
       Etapa 2: qual nível de canais performa melhor?  (livre=nível, fixo=W ótima, modelo=RF)
       Etapa 3: qual modelo performa melhor?         (livre=modelo, fixo=W ótima + nível ótimo)
```

## Design experimental — LOSO por contexto de crise

Cada contexto de crise (`ctx_id`) é uma unidade de leave-one-out: o modelo treina
em todos os outros contextos do paciente (undersample 1:1 pré/inter no treino) e é
testado no contexto deixado de fora, **sem undersampling no teste**.

## Correções aplicadas nesta versão

| # | Onde | Bug | Correção |
|---|------|-----|---------|
| 1 | `select_window_nested` | `all_contexts` usava UNIÃO de cp∪ci por janela, incluindo ctx_ids que existem só em cp ou só em ci para determinadas janelas → folds com `Xi_te=[]` → `None` silencioso | Trocar por UNIÃO das **interseções** por janela |
| 2 | `_train_eval_single` | `return None` quando treino ou teste vazio apagava o fold_ctx do CSV → N_Ctx reportado < real | Retornar dict com nan + campo `erro` em vez de None |
| 3 | `select_window_nested` | Pacientes com exatamente 2 ctx eram silenciosamente excluídos (exige ≥3) → sem `PATIENT_WINDOW` → pulados nas Etapas 2 e 3 | Fallback para LOSO direto com janela default |
| 4 | `show_stage_by_dataset` | `N_Ctx_Total` calculado via `fold_ctx.nunique()` — dependia dos folds que completaram, não dos contextos reais do paciente | Carregar `pipeline_manifest.json` para N_Ctx_Total canônico |

## Melhorias aplicadas

- **Z-score por contexto** em `load_patient_features` (remove drift inter-sessão)
- **Múltiplas seeds** no undersample (`N_SEEDS = 5`, seeds 42-46)


In [1]:
import subprocess, sys
for pkg in ['xgboost', 'scikit-learn', 'tqdm']:
    try:
        __import__(pkg.replace('scikit-learn','sklearn'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('Dependências OK.')

Dependências OK.


## 1. Imports e configuração

In [2]:
import json, warnings
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix
from xgboost import XGBClassifier

try:
    from tqdm.auto import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False
    def tqdm(it, **kw): return it

warnings.filterwarnings('ignore')

ROOT     = Path('data')
FEAT_DIR = ROOT / 'features'
PRE_EST  = ROOT / 'preictal_estimate.json'
MANIFEST = ROOT / 'pipeline_manifest.json'
OUT_DIR  = ROOT / 'results'
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_FEATS_CH = 19

# ── Janelas experimentais (Etapa 1) ───────────────────────────────────────────
WINDOWS_FIXED = {
    'W1':  8 * 60,
    'W2': 10 * 60,
    'W3': 13 * 60,
    'W4': 19 * 60,
}
# W5 = PRE_SEC individual do paciente (lido do preictal_estimate.json)

# ── Melhorias ─────────────────────────────────────────────────────────────────
ZSCORE_BY_CTX = True    # z-score por contexto antes de empilhar
N_SEEDS       = 5       # múltiplas seeds no undersample
SEEDS         = [42, 43, 44, 45, 46]

# Janela default para pacientes com apenas 2 contextos (sem seleção aninhada)
W_DEFAULT_S   = 13 * 60  # W3 = mediana típica dos resultados

print('Configuração carregada.')
print(f'  ZSCORE_BY_CTX = {ZSCORE_BY_CTX}')
print(f'  N_SEEDS       = {N_SEEDS}  (seeds {SEEDS})')
print(f'  W_DEFAULT_S   = {W_DEFAULT_S//60}min (fallback para pacientes com 2 ctx)')

Configuração carregada.
  ZSCORE_BY_CTX = True
  N_SEEDS       = 5  (seeds [42, 43, 44, 45, 46])
  W_DEFAULT_S   = 13min (fallback para pacientes com 2 ctx)


c:\Users\danil\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Classificador de Distância Mínima (MDC)

In [3]:
class MinimumDistanceClassifier:
    """Classificador de distância mínima ao centróide de classe.
    Compatível com a interface sklearn (fit/predict/predict_proba).
    """
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.centroids_ = {c: X[y == c].mean(axis=0) for c in self.classes_}
        return self

    def _dists(self, X):
        return np.column_stack([
            np.linalg.norm(X - self.centroids_[c], axis=1) for c in self.classes_
        ])

    def predict(self, X):
        d = self._dists(X)
        return self.classes_[np.argmin(d, axis=1)]

    def predict_proba(self, X):
        d = self._dists(X)
        inv = 1.0 / (d + 1e-10)
        p = inv / inv.sum(axis=1, keepdims=True)
        order = np.argsort(self.classes_)
        return p[:, order]

print('MinimumDistanceClassifier definido.')

MinimumDistanceClassifier definido.


## 3. Fábricas de modelos

In [4]:
def make_rf():
    return RandomForestClassifier(n_estimators=200, max_depth=12,
                                  random_state=42, n_jobs=-1,
                                  class_weight='balanced')

def make_xgb():
    return XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                         random_state=42, n_jobs=-1,
                         eval_metric='logloss', verbosity=0)

def make_svm():
    return SVC(kernel='rbf', C=1.0, probability=True,
               random_state=42, class_weight='balanced')

def make_lr():
    return LogisticRegression(max_iter=2000, random_state=42,
                              class_weight='balanced')

def make_nb():
    return GaussianNB()

def make_knn(k):
    return KNeighborsClassifier(n_neighbors=k)

def make_mdc():
    return MinimumDistanceClassifier()

MODELS = {
    'RF':   make_rf,
    'XGB':  make_xgb,
    'SVM':  make_svm,
    'LR':   make_lr,
    'NB':   make_nb,
    'kNN3': lambda: make_knn(3),
    'kNN5': lambda: make_knn(5),
    'kNN7': lambda: make_knn(7),
    'MDC':  make_mdc,
}

print(f'Modelos definidos: {list(MODELS.keys())}')

Modelos definidos: ['RF', 'XGB', 'SVM', 'LR', 'NB', 'kNN3', 'kNN5', 'kNN7', 'MDC']


## 4. Carregamento dos dados por paciente

**Melhoria aplicada:** z-score por contexto antes de empilhar — remove drift
inter-sessão (impedância, ganho do amplificador) que o `StandardScaler` do fold
não conseguia corrigir sozinho.

In [5]:
def load_patient_features(fp, zscore_by_ctx=ZSCORE_BY_CTX):
    """Carrega .npz do NB3 e retorna dict com arrays + meta.
    Se zscore_by_ctx=True, normaliza cada contexto individualmente
    antes de empilhar (remove drift inter-sessão).
    """
    raw  = np.load(fp, allow_pickle=True)
    meta = json.loads(str(raw['meta']))

    X_pre         = raw['X_pre'].copy().astype(np.float32)
    X_inter       = raw['X_inter'].copy().astype(np.float32)
    t_pre         = raw['t_pre'].copy()
    t_inter       = raw['t_inter'].copy()
    ctx_ids_pre   = raw['ctx_ids_pre'].copy()
    ctx_ids_inter = raw['ctx_ids_inter'].copy()

    if zscore_by_ctx:
        # ── Z-score por contexto: cada ctx tem sua própria média e std ──────
        # Isso remove diferenças de amplitude entre sessões de gravação
        # (impedância diferente, ganho do amplificador) sem tocar nas
        # diferenças reais entre pré-ictal e interictal DENTRO de cada ctx.
        for ctx_id in np.unique(ctx_ids_pre):
            mask = ctx_ids_pre == ctx_id
            if mask.sum() < 2: continue
            mu = X_pre[mask].mean(axis=0, keepdims=True)
            sd = X_pre[mask].std(axis=0, keepdims=True) + 1e-10
            X_pre[mask] = (X_pre[mask] - mu) / sd

        for ctx_id in np.unique(ctx_ids_inter):
            mask = ctx_ids_inter == ctx_id
            if mask.sum() < 2: continue
            mu = X_inter[mask].mean(axis=0, keepdims=True)
            sd = X_inter[mask].std(axis=0, keepdims=True) + 1e-10
            X_inter[mask] = (X_inter[mask] - mu) / sd

    return {
        'X_pre': X_pre, 'X_inter': X_inter,
        't_pre': t_pre, 't_inter': t_inter,
        'ctx_ids_pre': ctx_ids_pre, 'ctx_ids_inter': ctx_ids_inter,
        'meta': meta,
    }


# ── Carrega N_Ctx_Total canônico do manifesto (não depende de folds rodados) ──
N_CTX_CANONICAL = {}   # {(ds, pat): n_contextos_validos}
if MANIFEST.exists():
    with open(MANIFEST, encoding='utf-8') as f:
        manifest = json.load(f)
    for c in manifest.get('contexts', []):
        if c.get('npz_saved', False):
            key = (c['dataset'], c['paciente'])
            N_CTX_CANONICAL[key] = N_CTX_CANONICAL.get(key, 0) + 1
    print(f'Manifesto carregado: {len(N_CTX_CANONICAL)} pacientes, '
          f'{sum(N_CTX_CANONICAL.values())} contextos válidos')
else:
    print('AVISO: pipeline_manifest.json não encontrado — N_Ctx_Total virá dos CSVs')


# ── Carrega estimativas individuais de PRE_SEC (janela W5) ────────────────────
PRE_SEC_BY_PATIENT = {}
if PRE_EST.exists():
    with open(PRE_EST, encoding='utf-8') as f:
        pre_est_raw = json.load(f)
    for entry in pre_est_raw:
        key = (entry['dataset'], entry['paciente'])
        if entry.get('pre_sec') is not None:
            PRE_SEC_BY_PATIENT[key] = float(entry['pre_sec'])
    print(f'PRE_SEC individual carregado para {len(PRE_SEC_BY_PATIENT)} pacientes.')
else:
    print('AVISO: preictal_estimate.json não encontrado — W5 não será usada')


# ── Carrega todos os pacientes ─────────────────────────────────────────────────
PATIENTS_DATA = {}
for fp in sorted(FEAT_DIR.glob('*.npz')):
    d = load_patient_features(fp)
    ds, pat = d['meta']['dataset'], d['meta']['paciente']
    PATIENTS_DATA[(ds, pat)] = d

print(f'\n{len(PATIENTS_DATA)} pacientes carregados de {FEAT_DIR}')
for ds in sorted(set(k[0] for k in PATIENTS_DATA)):
    n = sum(1 for k in PATIENTS_DATA if k[0] == ds)
    print(f'  {ds}: {n} pacientes')

Manifesto carregado: 24 pacientes, 104 contextos válidos
PRE_SEC individual carregado para 18 pacientes.

24 pacientes carregados de data\features
  CHBMIT: 7 pacientes
  Mendeley: 3 pacientes
  SeizeIT2: 7 pacientes
  Siena: 7 pacientes


## 5. Funções centrais — filtro, LOSO, seleção de janela

### Correções nesta seção

**Correção 1** (`select_window_nested`): `all_contexts` agora usa UNIÃO das
**interseções** cp∩ci por janela. Antes usava `cp∪ci`, o que incluía ctx_ids sem
dados interictais para determinadas janelas — esses folds retornavam `None`
silenciosamente, encolhendo o N_Ctx reportado.

**Correção 2** (`_train_eval_single`): retorna `dict` com `nan` (nunca `None`)
quando treino ou teste está vazio. Assim o `fold_ctx` sempre aparece no CSV,
mesmo sem métricas válidas.

**Correção 3** (`select_window_nested` + chamada): pacientes com exatamente
2 contextos recebem fallback para `run_loso_patient` com `W_DEFAULT_S`.

**Melhoria** (`_train_eval_single`): loop de `N_SEEDS` seeds no undersample;
métricas do fold = média das seeds.

In [6]:
def filter_window(d, W_s, level_cols=None):
    """Filtra X_pre/X_inter pelos últimos W_s segundos.
    t_pre e t_inter são negativos (referencial do fim do segmento — ver NB3).
    level_cols: None = todas as colunas; int = primeiras level_cols colunas (nível aninhado).
    """
    t_pre, t_inter = d['t_pre'], d['t_inter']
    X_pre, X_inter = d['X_pre'], d['X_inter']
    cid_pre, cid_inter = d['ctx_ids_pre'], d['ctx_ids_inter']

    mask_p = t_pre   >= -W_s
    mask_i = t_inter >= -W_s

    Xp, Xi = X_pre[mask_p], X_inter[mask_i]
    cp, ci = cid_pre[mask_p], cid_inter[mask_i]

    if level_cols is not None:
        n_cols = min(level_cols, Xp.shape[1], Xi.shape[1])
        Xp, Xi = Xp[:, :n_cols], Xi[:, :n_cols]

    return Xp, Xi, cp, ci


def get_level_cols(d, level):
    """Nº de colunas para um nível (R0..R5), respeitando disponibilidade do paciente."""
    slices = d['meta']['level_slices']
    if level in slices:
        return slices[level]
    return max(slices.values())


def _train_eval_single(Xp, Xi, cp, ci, train_contexts, test_context, model_name):
    """Treina em train_contexts (undersample 1:1, N_SEEDS seeds) e avalia
    em test_context (distribuição real, sem undersampling).

    CORREÇÃO 2: retorna dict com nan em vez de None quando dados ausentes,
    para que fold_ctx apareça no CSV mesmo em folds inválidos.

    MELHORIA: loop de N_SEEDS seeds; métricas = média das seeds válidas.
    """
    # ── Seleciona dados de treino ──────────────────────────────────────────────
    tr_p_mask = np.isin(cp, train_contexts)
    tr_i_mask = np.isin(ci, train_contexts)
    Xp_tr, Xi_tr = Xp[tr_p_mask], Xi[tr_i_mask]

    _base = {'fold_ctx': int(test_context), 'auc': np.nan, 'f1': np.nan,
             'sensitivity': np.nan, 'specificity': np.nan, 'fp_h': np.nan,
             'n_train': 0, 'n_test': 0}

    if len(Xp_tr) == 0 or len(Xi_tr) == 0:
        return {**_base, 'erro': 'treino_vazio'}  # CORREÇÃO 2

    # ── Seleciona dados de teste ───────────────────────────────────────────────
    te_p_mask = cp == test_context
    te_i_mask = ci == test_context
    Xp_te, Xi_te = Xp[te_p_mask], Xi[te_i_mask]

    if len(Xp_te) == 0 or len(Xi_te) == 0:
        return {**_base, 'n_train': len(Xp_tr)+len(Xi_tr), 'erro': 'teste_vazio'}  # CORREÇÃO 2

    X_test = np.vstack([Xp_te, Xi_te])
    y_test = np.concatenate([np.ones(len(Xp_te)), np.zeros(len(Xi_te))])

    X_test = np.nan_to_num(X_test, nan=0.0, posinf=1e6, neginf=-1e6)

    # ── Loop de seeds (MELHORIA) ──────────────────────────────────────────────
    n_min = min(len(Xp_tr), len(Xi_tr))
    seed_metrics = []

    for seed in SEEDS:
        rng   = np.random.RandomState(seed)
        idx_p = rng.choice(len(Xp_tr), n_min, replace=False)
        idx_i = rng.choice(len(Xi_tr), n_min, replace=False)

        X_train = np.vstack([Xp_tr[idx_p], Xi_tr[idx_i]])
        y_train = np.concatenate([np.ones(n_min), np.zeros(n_min)])
        X_train = np.nan_to_num(X_train, nan=0.0, posinf=1e6, neginf=-1e6)

        scaler    = StandardScaler()
        Xtr_s     = scaler.fit_transform(X_train)
        Xte_s     = scaler.transform(X_test)
        Xtr_s     = np.clip(np.nan_to_num(Xtr_s, nan=0., posinf=0., neginf=0.), -50, 50)
        Xte_s     = np.clip(np.nan_to_num(Xte_s, nan=0., posinf=0., neginf=0.), -50, 50)

        try:
            clf   = MODELS[model_name]()
            clf.fit(Xtr_s, y_train)
            y_pred = clf.predict(Xte_s)
            y_score = clf.predict_proba(Xte_s)[:, 1] if hasattr(clf,'predict_proba') else y_pred.astype(float)

            auc_ = roc_auc_score(y_test, y_score) if len(set(y_test)) > 1 else np.nan
            f1_  = f1_score(y_test, y_pred, zero_division=0)
            tn, fp_, fn, tp_ = confusion_matrix(y_test, y_pred, labels=[0,1]).ravel()
            sens_ = tp_/(tp_+fn) if (tp_+fn)>0 else np.nan
            spec_ = tn/(tn+fp_) if (tn+fp_)>0 else np.nan
            dur_h = (len(Xi_te) * 15.0) / 3600.0
            fph_  = fp_/dur_h if dur_h>0 else np.nan
            seed_metrics.append({'auc':auc_,'f1':f1_,'sensitivity':sens_,
                                  'specificity':spec_,'fp_h':fph_})
        except Exception as e:
            seed_metrics.append({'auc':np.nan,'f1':np.nan,'sensitivity':np.nan,
                                  'specificity':np.nan,'fp_h':np.nan,'erro_seed':str(e)})

    # Média das seeds válidas
    def _mean(key):
        vals = [m[key] for m in seed_metrics if not np.isnan(m.get(key, np.nan))]
        return float(np.mean(vals)) if vals else np.nan

    return {
        'fold_ctx':    int(test_context),
        'auc':         _mean('auc'),
        'f1':          _mean('f1'),
        'sensitivity': _mean('sensitivity'),
        'specificity': _mean('specificity'),
        'fp_h':        _mean('fp_h'),
        'n_train':     len(X_train),
        'n_test':      len(X_test),
        'n_seeds_ok':  sum(1 for m in seed_metrics if not np.isnan(m.get('auc', np.nan))),
    }


def _run_loso_generic(Xp, Xi, cp, ci, contexts, model_name):
    """LOSO restrito a um subconjunto de contextos."""
    results = []
    for held_out in contexts:
        train_contexts = [c for c in contexts if c != held_out]
        if not train_contexts:
            continue
        r = _train_eval_single(Xp, Xi, cp, ci, train_contexts, held_out, model_name)
        if r is not None:
            results.append(r)
    return results


def run_loso_patient(d, W_s, level=None, model_name='RF'):
    """LOSO por contexto de crise para um único paciente (janela/nível fixos)."""
    level_cols = get_level_cols(d, level) if level else None
    Xp, Xi, cp, ci = filter_window(d, W_s, level_cols)
    if len(Xp) == 0 or len(Xi) == 0:
        return []
    # Apenas ctx com dados nas DUAS classes (interseção)
    contexts = sorted(set(cp) & set(ci))
    if len(contexts) < 2:
        return []
    return _run_loso_generic(Xp, Xi, cp, ci, contexts, model_name)


def select_window_nested(d, candidate_windows, level=None, model_name='RF'):
    """Seleção de janela por paciente SEM vazamento (LOSO aninhado).

    CORREÇÃO 1: all_contexts = UNIÃO das interseções cp∩ci por janela.
    Antes usava cp∪ci, incluindo ctx sem dados inter para certas janelas.

    CORREÇÃO 3 (fallback): se all_contexts < 3, usa run_loso_patient direto
    com a maior janela disponível (pacientes com 2 ctx).
    """
    level_cols = get_level_cols(d, level) if level else None
    cached = {}
    for wlabel, W_s in candidate_windows.items():
        Xp, Xi, cp, ci = filter_window(d, W_s, level_cols)
        cached[wlabel] = (Xp, Xi, cp, ci, W_s)

    # CORREÇÃO 1: UNIÃO das interseções (não UNIÃO das uniões)
    valid_per_window = [set(v[2]) & set(v[3]) for v in cached.values()]
    all_contexts = sorted(set().union(*valid_per_window))

    if len(all_contexts) < 2:
        return []

    # CORREÇÃO 3: paciente com exatamente 2 ctx → sem aninhamento, usa janela default
    if len(all_contexts) == 2:
        best_wlabel = max(candidate_windows, key=lambda k: candidate_windows[k])
        Xp, Xi, cp, ci, W_s = cached[best_wlabel]
        results = []
        for r in _run_loso_generic(Xp, Xi, cp, ci, all_contexts, model_name):
            r['chosen_window'] = best_wlabel
            r['chosen_W_s']    = W_s
            r['fallback_2ctx'] = True
            results.append(r)
        return results

    # Caso normal: >= 3 contextos → LOSO aninhado
    results = []
    for held_out in all_contexts:
        inner_contexts = [c for c in all_contexts if c != held_out]
        if len(inner_contexts) < 2:
            continue

        # Seleção interna: melhor janela usando só inner_contexts
        inner_auc = {}
        for wlabel, (Xp, Xi, cp, ci, W_s) in cached.items():
            # Filtra inner_contexts à interseção disponível para esta janela
            valid_inner = sorted(set(cp) & set(ci) & set(inner_contexts))
            if len(valid_inner) < 2:
                inner_auc[wlabel] = -np.inf
                continue
            inner_res = _run_loso_generic(Xp, Xi, cp, ci, valid_inner, model_name)
            aucs = [r['auc'] for r in inner_res if not np.isnan(r['auc'])]
            inner_auc[wlabel] = np.mean(aucs) if aucs else -np.inf

        if all(v == -np.inf for v in inner_auc.values()):
            continue
        best_wlabel = max(inner_auc, key=inner_auc.get)
        Xp, Xi, cp, ci, W_s = cached[best_wlabel]

        r = _train_eval_single(Xp, Xi, cp, ci, inner_contexts, held_out, model_name)
        if r is not None:
            r['chosen_window'] = best_wlabel
            r['chosen_W_s']    = W_s
            results.append(r)
    return results


print('Funções LOSO (corridas 1-3 + melhorias) definidas.')

Funções LOSO (corridas 1-3 + melhorias) definidas.


## 6. Etapa 1 — Qual janela W performa melhor? (LOSO aninhado, sem vazamento)

Janela escolhida **por fold** via LOSO interno, sem olhar o teste.
Fixo: nível máximo disponível por paciente, modelo=RF.

In [7]:
rows_s1 = []

for (ds, pat), d in tqdm(sorted(PATIENTS_DATA.items()), desc='Etapa 1 — janela'):
    candidate_windows = dict(WINDOWS_FIXED)
    w5 = PRE_SEC_BY_PATIENT.get((ds, pat))
    if w5 is not None:
        candidate_windows['W5'] = w5

    fold_results = select_window_nested(d, candidate_windows, level=None, model_name='RF')
    for r in fold_results:
        rows_s1.append({'dataset': ds, 'paciente': pat, **r})

df_s1 = pd.DataFrame(rows_s1)
df_s1.to_csv(OUT_DIR / 'stage1_windows.csv', index=False)
print(f'Etapa 1: {len(df_s1)} folds salvos em stage1_windows.csv')

Etapa 1 — janela: 100%|██████████| 24/24 [1:36:03<00:00, 240.14s/it]  

Etapa 1: 104 folds salvos em stage1_windows.csv


In [8]:
pd.set_option('display.width', 120)

agg_s1 = (df_s1.groupby('chosen_window')
               .agg(auc_mean=('auc','mean'), auc_std=('auc','std'),
                    f1_mean=('f1','mean'),
                    sens_mean=('sensitivity','mean'), spec_mean=('specificity','mean'),
                    fp_h_mean=('fp_h','mean'), n_folds=('fold_ctx','count'))
               .reset_index()
               .sort_values('auc_mean', ascending=False))

print('RANKING DE JANELAS (Etapa 1)')
print('='*100)
print(f'  {"Janela":<8}{"AUC":<16}{"F1":<8}{"Sens.":<8}{"Espec.":<8}{"FP/h":<8}{"n"}')
for _, row in agg_s1.iterrows():
    print(f'  {row["chosen_window"]:<8}'
          f'{row["auc_mean"]:.3f}±{row["auc_std"]:.3f}   '
          f'{row["f1_mean"]:<8.3f}{row["sens_mean"]:<8.3f}'
          f'{row["spec_mean"]:<8.3f}{row["fp_h_mean"]:<8.2f}{int(row["n_folds"])}')

print(f'\nGlobal: AUC={df_s1["auc"].mean():.3f}±{df_s1["auc"].std():.3f}  '
      f'F1={df_s1["f1"].mean():.3f}  Sens={df_s1["sensitivity"].mean():.3f}  '
      f'Espec={df_s1["specificity"].mean():.3f}  FP/h={df_s1["fp_h"].mean():.2f}')

PATIENT_WINDOW = (df_s1.groupby(['dataset','paciente'])['chosen_W_s'].median().to_dict())

print('\nJANELA CONSENSO POR PACIENTE')
print('='*70)
for (ds, pat), W_s in sorted(PATIENT_WINDOW.items()):
    n_ctx_real = N_CTX_CANONICAL.get((ds, pat), '?')
    n_ctx_fold = df_s1[(df_s1.dataset==ds)&(df_s1.paciente==pat)]['fold_ctx'].nunique()
    fallback   = df_s1[(df_s1.dataset==ds)&(df_s1.paciente==pat)].get('fallback_2ctx', pd.Series([False])).any()
    flag = ' [2ctx-fallback]' if fallback else ''
    moda = df_s1[(df_s1.dataset==ds)&(df_s1.paciente==pat)]['chosen_window'].value_counts().idxmax()
    print(f'  {ds:<10} {pat:<12} W={W_s/60:.1f}min  ({moda})  '
          f'folds={n_ctx_fold}/{n_ctx_real}{flag}')

RANKING DE JANELAS (Etapa 1)
  Janela  AUC             F1      Sens.   Espec.  FP/h    n
  W5      0.788±0.185   0.741   0.834   0.587   99.20   12
  W2      0.743±0.261   0.670   0.675   0.739   62.67   12
  W4      0.589±0.243   0.570   0.630   0.484   123.80  40
  W1      0.578±0.343   0.527   0.621   0.462   129.10  30
  W3      0.528±0.258   0.586   0.665   0.374   150.25  10

Global: AUC=0.620±0.281  F1=0.590  Sens=0.660  Espec=0.508  FP/h=117.98

JANELA CONSENSO POR PACIENTE
  CHBMIT     chb02        W=19.0min  (W4)  folds=2/2 [2ctx-fallback]
  CHBMIT     chb04        W=19.0min  (W4)  folds=2/2 [2ctx-fallback]
  CHBMIT     chb06        W=13.7min  (W5)  folds=7/7
  CHBMIT     chb07        W=8.0min  (W1)  folds=3/3
  CHBMIT     chb08        W=13.0min  (W3)  folds=4/4
  CHBMIT     chb10        W=13.0min  (W3)  folds=5/5
  CHBMIT     chb13        W=19.0min  (W4)  folds=2/2 [2ctx-fallback]
  Mendeley   p10          W=19.0min  (W4)  folds=2/2 [2ctx-fallback]
  Mendeley   p13          

## 7. Etapa 2 — Qual nível de canais performa melhor?

In [9]:
LEVELS = ['R0', 'R1', 'R2', 'R3', 'R4', 'R5']
rows_s2 = []

for (ds, pat), d in tqdm(sorted(PATIENTS_DATA.items()), desc='Etapa 2 — níveis'):
    W_s = PATIENT_WINDOW.get((ds, pat))
    if W_s is None:
        continue

    for level in LEVELS:
        for r in run_loso_patient(d, W_s, level=level, model_name='RF'):
            rows_s2.append({'dataset': ds, 'paciente': pat, 'level': level,
                            'W_s': W_s, **r})

df_s2 = pd.DataFrame(rows_s2)
df_s2.to_csv(OUT_DIR / 'stage2_levels.csv', index=False)
print(f'Etapa 2: {len(df_s2)} folds salvos em stage2_levels.csv')

Etapa 2 — níveis: 100%|██████████| 24/24 [22:26<00:00, 56.09s/it] 

Etapa 2: 624 folds salvos em stage2_levels.csv


In [10]:
agg_s2 = (df_s2.groupby('level')
               .agg(auc_mean=('auc','mean'), auc_std=('auc','std'),
                    f1_mean=('f1','mean'),
                    sens_mean=('sensitivity','mean'), spec_mean=('specificity','mean'),
                    fp_h_mean=('fp_h','mean'), n_folds=('fold_ctx','count'))
               .reset_index())
agg_s2['level'] = pd.Categorical(agg_s2['level'], categories=LEVELS, ordered=True)
agg_s2 = agg_s2.sort_values('level')

print('RANKING DE NÍVEIS (Etapa 2)')
print('='*100)
print(f'  {"Nível":<8}{"AUC":<16}{"F1":<8}{"Sens.":<8}{"Espec.":<8}{"FP/h":<8}{"n"}')
for _, row in agg_s2.iterrows():
    print(f'  {row["level"]:<8}'
          f'{row["auc_mean"]:.3f}±{row["auc_std"]:.3f}   '
          f'{row["f1_mean"]:<8.3f}{row["sens_mean"]:<8.3f}'
          f'{row["spec_mean"]:<8.3f}{row["fp_h_mean"]:<8.2f}{int(row["n_folds"])}')

BEST_LEVEL = agg_s2.loc[agg_s2['auc_mean'].idxmax(), 'level']
print(f'\nMelhor nível: {BEST_LEVEL}')

RANKING DE NÍVEIS (Etapa 2)
  Nível   AUC             F1      Sens.   Espec.  FP/h    n
  R0      0.629±0.287   0.616   0.694   0.491   122.16  104
  R1      0.621±0.289   0.597   0.668   0.500   120.08  104
  R2      0.652±0.268   0.615   0.681   0.530   112.72  104
  R3      0.657±0.266   0.616   0.681   0.535   111.50  104
  R4      0.657±0.269   0.614   0.680   0.535   111.53  104
  R5      0.652±0.270   0.611   0.676   0.529   113.10  104

Melhor nível: R4


## 8. Etapa 3 — Qual modelo performa melhor?

In [11]:
rows_s3 = []

for (ds, pat), d in tqdm(sorted(PATIENTS_DATA.items()), desc='Etapa 3 — modelos'):
    W_s = PATIENT_WINDOW.get((ds, pat))
    if W_s is None:
        continue

    level = 'R0' if ds == 'SeizeIT2' else BEST_LEVEL

    for model_name in MODELS:
        for r in run_loso_patient(d, W_s, level=level, model_name=model_name):
            rows_s3.append({'dataset': ds, 'paciente': pat, 'model': model_name,
                            'W_s': W_s, **r})

df_s3 = pd.DataFrame(rows_s3)
df_s3.to_csv(OUT_DIR / 'stage3_models.csv', index=False)
print(f'Etapa 3: {len(df_s3)} folds salvos em stage3_models.csv')

Etapa 3 — modelos:   0%|          | 0/24 [00:00<?, ?it/s]  File "c:\Users\danil\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\danil\AppData\Local\Programs\Python\Python313\Lib\subprocess.py", line 556, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\danil\AppData\Local\Programs\Python\Python313\Lib\subprocess.py", line 1038, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                        pass_fds, cwd, env,
                        ^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
                        gid, gids, uid, umask,
                        ^^^^^^^^^^^^^^^^^^^^^^
                 

Etapa 3: 936 folds salvos em stage3_models.csv


In [12]:
ALL_MODELS = ['RF', 'XGB', 'SVM', 'LR', 'NB', 'kNN3', 'kNN5', 'kNN7', 'MDC']

agg_s3 = (df_s3.groupby(['dataset','model'])
               .agg(auc_mean=('auc','mean'), auc_std=('auc','std'),
                    f1_mean=('f1','mean'), sens_mean=('sensitivity','mean'),
                    spec_mean=('specificity','mean'), fp_h_mean=('fp_h','mean'),
                    n_folds=('fold_ctx','count'))
               .reset_index())

print('COMPARAÇÃO DE MODELOS — AUC por dataset (Etapa 3)')
print('='*112)
header = f'  {"Dataset":<12} ' + ' '.join(f'{m:<10}' for m in ALL_MODELS) + '  Melhor'
print(header)
print('  ' + '-'*(len(header)-2))
for ds in sorted(agg_s3['dataset'].unique()):
    sub  = agg_s3[agg_s3['dataset']==ds].set_index('model')
    vals = {m: sub.loc[m,'auc_mean'] if m in sub.index else float('nan') for m in ALL_MODELS}
    best = max(vals, key=lambda k: vals[k] if not np.isnan(vals[k]) else -1)
    row  = f'  {ds:<12} ' + ' '.join(
        f'{vals[m]:<10.3f}' if not np.isnan(vals[m]) else f'{"n/a":<10}' for m in ALL_MODELS
    ) + f'  {best}'
    print(row)

print()
print('MÉTRICAS COMPLETAS POR MODELO (agregado global)')
print('='*90)
print(f'  {"Modelo":<8}{"AUC":<16}{"F1":<8}{"Sens.":<8}{"Espec.":<8}{"FP/h":<8}{"n"}')
gbl_full = (df_s3.groupby('model')
                 .agg(auc_mean=('auc','mean'), auc_std=('auc','std'),
                      f1_mean=('f1','mean'), sens_mean=('sensitivity','mean'),
                      spec_mean=('specificity','mean'), fp_h_mean=('fp_h','mean'),
                      n_folds=('fold_ctx','count'))
                 .reindex(ALL_MODELS)
                 .sort_values('auc_mean', ascending=False))
for m, row in gbl_full.iterrows():
    print(f'  {m:<8}'
          f'{row["auc_mean"]:.3f}±{row["auc_std"]:.3f}   '
          f'{row["f1_mean"]:<8.3f}{row["sens_mean"]:<8.3f}'
          f'{row["spec_mean"]:<8.3f}{row["fp_h_mean"]:<8.2f}{int(row["n_folds"])}')

gbl = df_s3.groupby('model')['auc'].mean()
best_global = gbl.dropna().idxmax()
print(f'\nMelhor modelo global: {best_global}  (AUC = {gbl[best_global]:.3f})')

COMPARAÇÃO DE MODELOS — AUC por dataset (Etapa 3)
  Dataset      RF         XGB        SVM        LR         NB         kNN3       kNN5       kNN7       MDC         Melhor
  -----------------------------------------------------------------------------------------------------------------------
  CHBMIT       0.673      0.649      0.621      0.544      0.594      0.634      0.646      0.651      0.578       RF
  Mendeley     0.700      0.634      0.730      0.663      0.703      0.634      0.675      0.685      0.671       SVM
  SeizeIT2     0.674      0.683      0.652      0.575      0.558      0.610      0.617      0.613      0.566       XGB
  Siena        0.575      0.591      0.607      0.497      0.536      0.625      0.632      0.626      0.501       kNN5

MÉTRICAS COMPLETAS POR MODELO (agregado global)
  Modelo  AUC             F1      Sens.   Espec.  FP/h    n
  RF      0.657±0.269   0.614   0.680   0.535   111.53  104
  XGB     0.655±0.254   0.617   0.671   0.552   107.52  104
 

## 9. Resumo final e exportação

In [13]:
summary = {
    'versao': 'NB4-corrigido-v2',
    'correcoes': ['select_window_nested-uniao-intersecoes',
                  '_train_eval_single-retorna-dict-nan',
                  'select_window_nested-fallback-2ctx',
                  'N_Ctx_Total-do-manifesto'],
    'melhorias': ['zscore_por_contexto', f'undersample_{N_SEEDS}_seeds'],
    'nested_window_selection': True,
    'best_level': str(BEST_LEVEL),
    'best_model_global': str(best_global),
    'best_model_auc_global': float(gbl[best_global]),
    'patient_windows_min': {f'{k[0]}/{k[1]}': round(v/60,1)
                            for k, v in PATIENT_WINDOW.items()},
    'stage1_ranking_by_chosen_window': agg_s1.to_dict(orient='records'),
    'stage2_ranking_by_level': agg_s2.to_dict(orient='records'),
    'stage3_ranking_by_dataset_model': agg_s3.to_dict(orient='records'),
    'stage3_global_by_model': gbl_full.reset_index().to_dict(orient='records'),
}

SUMMARY_PATH = OUT_DIR / 'nb4_summary.json'
with open(SUMMARY_PATH, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False, default=str)

print('RESUMO FINAL')
print('='*50)
print(f'  Nível ótimo : {BEST_LEVEL}')
print(f'  Modelo ótimo: {best_global}  (AUC={gbl[best_global]:.3f})')
print(f'\nResultados em: {OUT_DIR}/')

try:
    from google.colab import files
    files.download(str(SUMMARY_PATH))
except ImportError:
    print(f'Arquivo local: {SUMMARY_PATH.resolve()}')

RESUMO FINAL
  Nível ótimo : R4
  Modelo ótimo: RF  (AUC=0.657)

Resultados em: data\results/
Arquivo local: D:\TCC\data\results\nb4_summary.json


## 10. Tabelas por paciente

**N_Ctx_Total** agora usa o manifesto do NB1 como fonte canônica (não o `fold_ctx.nunique()`
do CSV). Isso separa claramente "quantos contextos o paciente tem" de "quantos folds
completaram" — a coluna `N_Ctx` continua mostrando os folds completados.

In [14]:
from IPython.display import display, HTML

RES_DIR = Path('data/results')
pd.set_option('display.max_rows', None)

def show_stage_by_dataset(csv_name, group_col, title):
    fp = RES_DIR / csv_name
    if not fp.exists():
        print(f'{csv_name} não encontrado.')
        return
    df = pd.read_csv(fp)

    # CORREÇÃO 4: N_Ctx_Total do manifesto
    if N_CTX_CANONICAL:
        df['N_Ctx_Total'] = df.apply(
            lambda r: N_CTX_CANONICAL.get((r['dataset'], r['paciente']), None), axis=1)
    else:
        # Fallback: conta fold_ctx únicos por paciente (comportamento antigo)
        n_ctx_csv = df.groupby(['dataset','paciente'])['fold_ctx'].nunique().rename('N_Ctx_Total')
        df = df.merge(n_ctx_csv.reset_index(), on=['dataset','paciente'])

    agg = (df.groupby(['dataset', 'paciente', group_col])
             .agg(AUC=('auc','mean'), AUC_std=('auc','std'),
                  F1=('f1','mean'), Sens=('sensitivity','mean'),
                  Espec=('specificity','mean'), FP_h=('fp_h','mean'),
                  N_Ctx=('fold_ctx','nunique'),
                  N_Ctx_Total=('N_Ctx_Total','first'))
             .reset_index()
             .rename(columns={group_col: group_col.capitalize(), 'paciente': 'Paciente'}))

    display(HTML(f'<h3>{title}</h3>'))
    for ds in sorted(agg['dataset'].unique()):
        sub = (agg[agg['dataset']==ds]
                  .drop(columns='dataset')
                  .sort_values(['Paciente', group_col.capitalize()])
                  .reset_index(drop=True))
        cols = ['Paciente', 'N_Ctx_Total', group_col.capitalize(),
                'N_Ctx', 'AUC', 'AUC_std', 'F1', 'Sens', 'Espec', 'FP_h']
        sub = sub[[c for c in cols if c in sub.columns]]
        display(HTML(f'<h4>{ds}</h4>'))
        styled = (sub.style
                     .background_gradient(subset=['AUC'], cmap='RdYlGn', vmin=0.4, vmax=1.0)
                     .format({'AUC':'{:.3f}','AUC_std':'{:.3f}','F1':'{:.3f}',
                              'Sens':'{:.3f}','Espec':'{:.3f}','FP_h':'{:.2f}'})
                     .set_caption(ds))
        display(styled)


show_stage_by_dataset('stage1_windows.csv', 'chosen_window',
    'ETAPA 1 — Desempenho por paciente (janela escolhida via LOSO aninhado)')
show_stage_by_dataset('stage2_levels.csv', 'level',
    'ETAPA 2 — Desempenho por paciente (nível de canais)')
show_stage_by_dataset('stage3_models.csv', 'model',
    'ETAPA 3 — Desempenho por paciente (modelo)')

,Paciente,N_Ctx_Total,Chosen_window,N_Ctx,AUC,AUC_std,F1,Sens,Espec,FP_h
0,chb02,2,W4,2,0.438,0.183,0.446,0.361,0.663,80.84
1,chb04,2,W4,2,0.664,0.064,0.469,0.528,0.653,83.31
2,chb06,7,W1,1,0.478,nan,0.628,0.903,0.056,226.50
3,chb06,7,W5,6,0.716,0.225,0.761,0.863,0.553,107.26
4,chb07,3,W1,2,0.652,0.001,0.598,0.794,0.166,200.25
5,chb07,3,W2,1,0.988,nan,0.946,0.897,1.000,0.00
6,chb08,4,W1,1,0.694,nan,0.733,0.903,0.456,130.50
7,chb08,4,W3,2,0.682,0.284,0.693,0.743,0.563,104.77
8,chb08,4,W4,1,0.737,nan,0.730,0.976,0.311,165.47
9,chb10,5,W1,1,0.000,nan,0.152,0.168,0.000,240.00


,Paciente,N_Ctx_Total,Chosen_window,N_Ctx,AUC,AUC_std,F1,Sens,Espec,FP_h
0,p10,2,W4,2,0.522,0.003,0.504,0.573,0.457,130.24
1,p13,2,W4,2,0.531,0.373,0.685,0.780,0.461,129.28
2,p15,3,W1,1,0.993,nan,0.151,0.084,1.000,0.00
3,p15,3,W2,2,0.842,0.133,0.830,0.821,0.844,37.54


,Paciente,N_Ctx_Total,Chosen_window,N_Ctx,AUC,AUC_std,F1,Sens,Espec,FP_h
0,sub-002,5,W1,5,0.887,0.106,0.817,0.890,0.702,71.54
1,sub-035,8,W1,3,0.412,0.475,0.686,0.983,0.105,214.71
2,sub-035,8,W4,4,0.801,0.298,0.743,0.884,0.448,132.48
3,sub-035,8,W5,1,0.902,nan,0.728,0.995,0.259,177.82
4,sub-039,9,W1,7,0.496,0.338,0.407,0.474,0.476,125.64
5,sub-039,9,W2,1,0.317,nan,0.341,0.318,0.456,130.46
6,sub-039,9,W4,1,0.523,nan,0.458,0.437,0.528,113.28
7,sub-047,7,W2,1,0.358,nan,0.430,0.513,0.128,209.23
8,sub-047,7,W3,3,0.417,0.360,0.571,0.659,0.297,168.78
9,sub-047,7,W4,3,0.457,0.061,0.576,0.741,0.195,193.28


,Paciente,N_Ctx_Total,Chosen_window,N_Ctx,AUC,AUC_std,F1,Sens,Espec,FP_h
0,PN01,2,W4,2,0.302,0.004,0.460,0.563,0.121,211.04
1,PN05,2,W4,2,0.879,0.121,0.775,0.962,0.432,136.32
2,PN09,3,W1,1,0.220,nan,0.087,0.069,0.465,128.52
3,PN09,3,W3,2,0.600,0.236,0.580,0.621,0.451,131.76
4,PN10,4,W4,2,0.506,0.157,0.408,0.388,0.689,74.56
5,PN10,4,W5,2,0.769,0.091,0.484,0.472,0.651,83.79
6,PN13,3,W1,2,0.360,0.181,0.251,0.191,0.677,77.42
7,PN13,3,W4,1,0.345,nan,0.310,0.245,0.667,80.00
8,PN14,3,W2,1,0.428,nan,0.676,0.985,0.046,228.92
9,PN14,3,W4,2,0.733,0.268,0.719,0.951,0.272,174.61


,Paciente,N_Ctx_Total,Level,N_Ctx,AUC,AUC_std,F1,Sens,Espec,FP_h
0,chb02,2,R0,2,0.809,0.050,0.678,0.676,0.776,53.68
1,chb02,2,R1,2,0.659,0.021,0.522,0.516,0.691,74.21
2,chb02,2,R2,2,0.506,0.245,0.292,0.228,0.692,73.89
3,chb02,2,R3,2,0.519,0.147,0.307,0.264,0.689,74.53
4,chb02,2,R4,2,0.429,0.190,0.412,0.339,0.643,85.58
5,chb02,2,R5,2,0.438,0.183,0.446,0.361,0.663,80.84
6,chb04,2,R0,2,0.344,0.167,0.381,0.511,0.479,125.05
7,chb04,2,R1,2,0.600,0.019,0.375,0.520,0.509,117.73
8,chb04,2,R2,2,0.645,0.064,0.409,0.467,0.578,101.20
9,chb04,2,R3,2,0.649,0.040,0.409,0.469,0.582,100.23


...

,Paciente,N_Ctx_Total,Level,N_Ctx,AUC,AUC_std,F1,Sens,Espec,FP_h
0,PN01,2,R0,2,0.298,0.051,0.490,0.620,0.080,220.80
1,PN01,2,R1,2,0.275,0.051,0.429,0.508,0.143,205.59
2,PN01,2,R2,2,0.398,0.036,0.547,0.671,0.215,188.45
3,PN01,2,R3,2,0.388,0.029,0.552,0.679,0.217,187.84
4,PN01,2,R4,2,0.398,0.016,0.537,0.654,0.213,188.78
5,PN01,2,R5,2,0.302,0.004,0.460,0.563,0.121,211.04
6,PN05,2,R0,2,0.677,0.011,0.634,0.797,0.304,167.04
7,PN05,2,R1,2,0.755,0.059,0.678,0.843,0.364,152.64
8,PN05,2,R2,2,0.667,0.138,0.682,0.862,0.319,163.52
9,PN05,2,R3,2,0.732,0.085,0.708,0.903,0.323,162.56


,Paciente,N_Ctx_Total,Model,N_Ctx,AUC,AUC_std,F1,Sens,Espec,FP_h
0,chb02,2,LR,2,0.463,0.063,0.484,0.520,0.386,147.47
1,chb02,2,MDC,2,0.474,0.209,0.481,0.532,0.408,142.11
2,chb02,2,NB,2,0.411,0.155,0.423,0.383,0.625,90.00
3,chb02,2,RF,2,0.429,0.190,0.412,0.339,0.643,85.58
4,chb02,2,SVM,2,0.247,0.029,0.333,0.348,0.271,174.95
5,chb02,2,XGB,2,0.556,0.047,0.446,0.420,0.542,109.89
6,chb02,2,kNN3,2,0.276,0.035,0.288,0.316,0.279,173.05
7,chb02,2,kNN5,2,0.263,0.029,0.283,0.308,0.254,179.05
8,chb02,2,kNN7,2,0.249,0.013,0.286,0.313,0.237,183.16
9,chb04,2,LR,2,0.403,0.097,0.461,0.541,0.310,165.64


,Paciente,N_Ctx_Total,Model,N_Ctx,AUC,AUC_std,F1,Sens,Espec,FP_h
0,p10,2,LR,2,0.729,0.220,0.692,0.720,0.633,88.00
1,p10,2,MDC,2,0.755,0.090,0.703,0.687,0.753,59.20
2,p10,2,NB,2,0.653,0.216,0.719,0.960,0.267,176.00
3,p10,2,RF,2,0.610,0.075,0.511,0.567,0.496,120.96
4,p10,2,SVM,2,0.754,0.249,0.730,0.867,0.440,134.40
5,p10,2,XGB,2,0.434,0.329,0.419,0.493,0.440,134.40
6,p10,2,kNN3,2,0.732,0.179,0.749,0.847,0.553,107.20
7,p10,2,kNN5,2,0.760,0.156,0.757,0.833,0.627,89.60
8,p10,2,kNN7,2,0.776,0.162,0.767,0.827,0.667,80.00
9,p13,2,LR,2,0.421,0.099,0.472,0.587,0.407,142.40


,Paciente,N_Ctx_Total,Model,N_Ctx,AUC,AUC_std,F1,Sens,Espec,FP_h
0,sub-002,5,LR,5,0.841,0.190,0.791,0.819,0.742,61.94
1,sub-002,5,MDC,5,0.805,0.275,0.560,0.484,0.877,29.42
2,sub-002,5,NB,5,0.714,0.283,0.257,0.187,0.877,29.42
3,sub-002,5,RF,5,0.887,0.106,0.817,0.890,0.702,71.54
4,sub-002,5,SVM,5,0.843,0.250,0.808,0.813,0.768,55.74
5,sub-002,5,XGB,5,0.894,0.093,0.827,0.858,0.768,55.74
6,sub-002,5,kNN3,5,0.800,0.187,0.724,0.652,0.839,38.71
7,sub-002,5,kNN5,5,0.809,0.219,0.708,0.639,0.832,40.26
8,sub-002,5,kNN7,5,0.809,0.219,0.726,0.658,0.858,34.06
9,sub-035,8,LR,8,0.759,0.244,0.736,0.768,0.622,90.60


,Paciente,N_Ctx_Total,Model,N_Ctx,AUC,AUC_std,F1,Sens,Espec,FP_h
0,PN01,2,LR,2,0.391,0.067,0.484,0.601,0.138,206.82
1,PN01,2,MDC,2,0.288,0.064,0.341,0.355,0.271,174.99
2,PN01,2,NB,2,0.379,0.108,0.494,0.574,0.245,181.31
3,PN01,2,RF,2,0.398,0.016,0.537,0.654,0.213,188.78
4,PN01,2,SVM,2,0.301,0.097,0.434,0.501,0.188,194.82
5,PN01,2,XGB,2,0.510,0.231,0.545,0.589,0.440,134.40
6,PN01,2,kNN3,2,0.302,0.104,0.375,0.387,0.319,163.41
7,PN01,2,kNN5,2,0.288,0.092,0.373,0.388,0.303,167.20
8,PN01,2,kNN7,2,0.280,0.091,0.364,0.375,0.303,167.20
9,PN05,2,LR,2,0.531,0.123,0.509,0.476,0.619,91.52
